# Multi-Agent · Day 35 — **Google ADK & the A2A Protocol**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2 hrs

Days 31–34 kept every agent inside one framework and one process. Reality at a bank is messier: a fraud team ships an agent on LangGraph, a risk team ships one on CrewAI, a vendor exposes one you can't see inside at all. **Agent2Agent (A2A)** is Google's open protocol for exactly this — a wire format so agents built on *different* stacks can discover and call each other. **ADK** (Agent Development Kit) is Google's framework for building A2A-speaking agents.

The two ideas you must own for an AVP interview: the **Agent Card** (how an agent advertises itself) and the **task lifecycle** (how one agent delegates to another over HTTP). Both are plain JSON — we build runnable mocks of each below, no cloud account needed.

Today:
1. The **Agent Card** — an agent's public, machine-readable capability sheet.
2. A **registry + discovery** — how a client finds the right agent for a task.
3. The **A2A task lifecycle** — submit → working → completed, with typed states.
4. **Cross-framework interop** — why A2A is what lets your LangGraph agent call a CrewAI one.

## 1. The Agent Card

Every A2A agent serves a card at `/.well-known/agent.json`. It's the contract: name, skills, endpoint, auth, and I/O modes. A client reads the card *before* sending work — same idea as an OpenAPI spec, but for an agent.

In [1]:
from dataclasses import dataclass, field, asdict
import json

@dataclass
class Skill:
    id: str
    description: str
    input_modes: list[str] = field(default_factory=lambda: ["text"])
    output_modes: list[str] = field(default_factory=lambda: ["text"])

@dataclass
class AgentCard:
    """Mirrors the A2A AgentCard served at /.well-known/agent.json."""
    name: str
    description: str
    url: str
    version: str
    skills: list[Skill]
    auth_schemes: list[str] = field(default_factory=lambda: ["bearer"])

sanctions_card = AgentCard(
    name="sanctions-screening-agent",
    description="Screens a party against consolidated sanctions lists.",
    url="https://agents.internal.bank/sanctions",
    version="1.2.0",
    skills=[Skill("screen_party", "Return sanctions hits for a name + country")],
)
print(json.dumps(asdict(sanctions_card), indent=2))

{
  "name": "sanctions-screening-agent",
  "description": "Screens a party against consolidated sanctions lists.",
  "url": "https://agents.internal.bank/sanctions",
  "version": "1.2.0",
  "skills": [
    {
      "id": "screen_party",
      "description": "Return sanctions hits for a name + country",
      "input_modes": [
        "text"
      ],
      "output_modes": [
        "text"
      ]
    }
  ],
  "auth_schemes": [
    "bearer"
  ]
}


☝ The card is **data, not code**, and that's the point — a client on any language or framework parses JSON. `skills` is what a caller matches against ("I need `screen_party`"); `auth_schemes` tells it how to authenticate; `url` is where tasks go. In production this is served over HTTPS behind the bank's mTLS, and the card itself carries no secrets — it's a public advertisement.

## 2. Registry + discovery

With many agents, a client needs to *find* the one with the right skill. A2A supports a registry (or a Google Agent Engine directory). Discovery = match required skill → cards that offer it.

In [2]:
class AgentRegistry:
    """Discovery layer: register cards, look them up by skill id."""
    def __init__(self):
        self._cards: dict[str, AgentCard] = {}

    def register(self, card: AgentCard) -> None:
        self._cards[card.name] = card

    def find_by_skill(self, skill_id: str) -> list[AgentCard]:
        return [c for c in self._cards.values()
                if any(s.id == skill_id for s in c.skills)]

registry = AgentRegistry()
registry.register(sanctions_card)
registry.register(AgentCard(
    name="credit-scoring-agent", description="PD/LGD scoring for SME facilities.",
    url="https://agents.internal.bank/credit", version="0.9.1",
    skills=[Skill("score_facility", "Return PD and suggested limit")]))

matches = registry.find_by_skill("screen_party")
print("agents offering screen_party:", [c.name for c in matches])
print("endpoint:", matches[0].url)

agents offering screen_party: ['sanctions-screening-agent']
endpoint: https://agents.internal.bank/sanctions


☝ Discovery decouples the caller from the callee's location and framework — the orchestrator asks for a *capability* (`screen_party`), not a hard-coded hostname. That's the difference between a brittle integration ("call sanctions-svc-prod-3") and a resilient one ("find something that can screen a party"). Swap the agent's implementation, keep the skill id, and every caller keeps working.

## 3. The A2A task lifecycle

A2A work is a **task** with a state machine, not a fire-and-forget call — because agent work is long-running (it may pause for input or a human). States: `submitted → working → (input-required) → completed | failed`.

In [3]:
from enum import StrEnum
import uuid, time

class TaskState(StrEnum):
    SUBMITTED = "submitted"
    WORKING = "working"
    INPUT_REQUIRED = "input-required"
    COMPLETED = "completed"
    FAILED = "failed"

@dataclass
class Task:
    id: str
    skill: str
    params: dict
    state: TaskState = TaskState.SUBMITTED
    artifacts: list[dict] = field(default_factory=list)
    history: list[str] = field(default_factory=list)

    def transition(self, to: TaskState, note: str = "") -> None:
        self.history.append(f"{self.state} -> {to} {note}".strip())
        self.state = to

# A mock A2A server for the sanctions agent: it *is* the skill implementation.
SANCTIONS = {"Vulpes SA": [{"list": "OFAC", "match": 0.97}]}

def handle_task(task: Task) -> Task:
    task.transition(TaskState.WORKING)
    name = task.params.get("name", "")
    hits = SANCTIONS.get(name, [])
    task.artifacts.append({"name": name, "hits": hits})
    task.transition(TaskState.COMPLETED, f"({len(hits)} hit(s))")
    return task

t = Task(id=str(uuid.uuid4())[:8], skill="screen_party", params={"name": "Vulpes SA"})
handle_task(t)
print("state:", t.state)
print("artifact:", t.artifacts[0])
print("audit:", *t.history, sep="\n  ")

state: completed
artifact: {'name': 'Vulpes SA', 'hits': [{'list': 'OFAC', 'match': 0.97}]}
audit:
  submitted -> working
  working -> completed (1 hit(s))


☝ The state machine is what makes A2A safe for **long-running, human-gated** bank work: a task can park in `input-required` (waiting on an analyst) and resume — the client polls or subscribes rather than blocking a thread for ten minutes. Note `history` and `artifacts`: every task carries its own audit trail and typed outputs, so a regulator can reconstruct exactly what any agent did and returned. This is the same auditability instinct as LangGraph's `messages` reducer, standardised across vendors.

## 4. Cross-framework interop — the whole reason A2A exists

Now the payoff: a **LangGraph** supervisor node delegating a sub-task to the sanctions agent — which could be ADK, CrewAI, or a closed vendor. The supervisor neither knows nor cares; it speaks A2A.

In [4]:
def a2a_client_call(registry: AgentRegistry, skill: str, params: dict) -> dict:
    """What a LangGraph node calls. Framework-agnostic: card in, artifact out."""
    cards = registry.find_by_skill(skill)
    if not cards:
        return {"error": f"no agent offers {skill}"}
    card = cards[0]                                   # + auth via card.auth_schemes
    task = Task(id=str(uuid.uuid4())[:8], skill=skill, params=params)
    result = handle_task(task)                        # real A2A: HTTP POST to card.url
    return {"agent": card.name, "state": result.state, "artifacts": result.artifacts}

# --- inside a LangGraph supervisor, this is just another node ---
def compliance_node(state: dict) -> dict:
    out = a2a_client_call(registry, "screen_party", {"name": state["counterparty"]})
    hit = bool(out["artifacts"][0]["hits"])
    return {"sanctions_hit": hit, "delegated_to": out["agent"]}

print(compliance_node({"counterparty": "Vulpes SA"}))
print(compliance_node({"counterparty": "ACME Ltd"}))

{'sanctions_hit': True, 'delegated_to': 'sanctions-screening-agent'}
{'sanctions_hit': False, 'delegated_to': 'sanctions-screening-agent'}


☝ `compliance_node` is a normal LangGraph node — but the work happens in a *different* agent, possibly a different framework, possibly owned by another team or vendor. That's the strategic bet: A2A is to agents what REST was to services. **Why it matters for Barclays:** you don't rewrite the fraud team's CrewAI agent to use it — you consume its Agent Card. Enterprises adopt A2A precisely so agents built at different times, by different teams, on different stacks still compose.

## 5. Recap

| Concept | ADK / A2A mechanism | Cell |
|---|---|---|
| Advertise capabilities | Agent Card at `/.well-known/agent.json` | 1 |
| Find the right agent | registry / directory, match by skill id | 2 |
| Long-running work | Task state machine (submitted→working→completed) | 3 |
| Audit + typed output | task `history` + `artifacts` | 3 |
| Cross-framework call | A2A client — card in, artifact out | 4 |

**One-liner for the interview:** *MCP connects an agent to its tools; A2A connects an agent to other agents.* You'll build the MCP half in Module 6. Tomorrow (Day 36): shared memory and task delegation *within* your own multi-agent system — Redis-backed state, task queues, and agent-to-agent sub-questions.